In [ ]:
import sys
import os
import yaml

In [ ]:
import rich

In [ ]:
notebook_dir = os.getcwd()
src_dir = notebook_dir  # The notebook is already in src/
sys.path.insert(0, src_dir)

print(f"Added to path: {src_dir}")
print(f"Current working directory: {notebook_dir}")

In [ ]:
from app.trainer import OnPolicyTrainer, OnPolicySchedule

# Actor Critic

In [ ]:
from app.rl_agents import ActorCritic
from app.models import ValueModel, StochasticDiscretePolicy, StochasticContinuousPolicy
from app.schedulers import ScheduleWrapper
from app.normalizer import RunningNorm, BatchNorm
from app.env_wrapper import GymnasiumWrapper
from app.buffer import RolloutBuffer
from app.renderer import Renderer
from app.rl_callbacks import WandbCallback
from app.logging_config import configure_logging

In [ ]:
import numpy as np
import torch as T
np.__version__

In [ ]:
# Create Env
env = GymnasiumWrapper(
    cfg='LunarLanderContinuous-v3',
    num_envs=8,
    wrappers=[],
    render_mode=None,
    seed=42,
    obs_key=None,
    goal_key=None,
    ach_goal_key=None
    )

In [ ]:
env.env.num_envs

In [ ]:
env.observation_space.sample()

In [ ]:
# Create policy
policy = StochasticDiscretePolicy(
    env=env,
    layer_config=[
        {
            'type': 'dense',
            'params': {'units': 64, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        },
        {
            'type': 'dense',
            'params': {'units': 32, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    output_config=[
        {
            'type': 'dense',
            'params': {'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    optimizer_params={
        'type': 'Adam',
        'params': {'lr': 0.001}
    },
    lr_scheduler=None,
    distribution='categorical',
    device='cuda'
)

In [ ]:
# Create policy
policy = StochasticContinuousPolicy(
    env=env,
    layer_config=[
        {
            'type': 'dense',
            'params': {'units': 64, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        },
        {
            'type': 'dense',
            'params': {'units': 32, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    output_config=[
        {
            'type': 'dense',
            'params': {'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    optimizer_params={
        'type': 'Adam',
        'params': {'lr': 0.001}
    },
    lr_scheduler=None,
    distribution='normal',
    device='cuda'
)

In [ ]:
# Create value model
value = ValueModel(
    env=env,
    layer_config=[
        {
            'type': 'dense',
            'params': {'units': 64, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        },
        {
            'type': 'dense',
            'params': {'units': 32, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    output_config=[
        {
            'type': 'dense',
            'params': {'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    optimizer_params={
        'type': 'Adam',
        'params': {'lr': 0.0001}
    },
    lr_scheduler=None,
    device='cuda'
)

In [ ]:
obs = env.reset()

In [ ]:
dist = policy(env.sample_observation().current_states)

In [ ]:
actions =T.tensor(env.env.action_space.sample(), device='cuda')

In [ ]:
actions

In [ ]:
actions.view(-1)

In [ ]:
dist.log_prob(actions)

In [ ]:
# Create Normalizers
state_normalizer = Normalizer(
    size=4,
    device='cuda'
)

advantage_normalizer = Normalizer(
    size=1,
    device='cuda'
)

In [ ]:
# Set params
discount = 0.99
policy_trace_decay = 0.0
value_trace_decay = 0.0
entropy_coefficient = 0.0
entropy_schedule = None
gae_coefficient = 0.95
save_dir = "E:/Documents/Programming/Projects/Reinforcement/PhoenX_RL/src/Trained_Models/Test_CartPole-v1_1/Reinforce/"
device = "cuda"

In [ ]:
# Build ActorCritic
agent = ActorCritic(
    policy=policy,
    value=value,
    discount=discount,
    policy_trace_decay=policy_trace_decay,
    value_trace_decay=value_trace_decay,
    entropy_coefficient=entropy_coefficient,
    # entropy_schedule=entropy_schedule,
    gae_coefficient=gae_coefficient,
    state_normalizer=state_normalizer,
    advantage_normalizer=advantage_normalizer,
    save_dir=save_dir,
    device=device
)


In [ ]:
# Create Buffer
buffer = RolloutBuffer(
    env=env,
    buffer_size=100,
    device=device
)

In [ ]:
# Create Schedule
schedule = OnPolicySchedule(
    unit='episode',
    num_units=1000,
    learn_unit='timestep',
    num_learn_units=800,
    seed=42
)


In [ ]:
# Create Renderer
renderer = Renderer(
    render_freq=1000,
    save_dir=save_dir,
    fps=30,
    codec='libx264'
)


In [ ]:
# Create callbacks
callbacks = [
    WandbCallback(
        project_name="CartPole-v1"
    )
]

In [ ]:
# Create trainer
trainer = OnPolicyTrainer(
    agent=agent,
    env=env,
    buffer=buffer,
    schedule=schedule,
    renderer=renderer,
    callbacks=callbacks
)

In [ ]:
trainer.train()

In [ ]:
trainer.buffer.states.shape

# Test agent.py

In [ ]:
from scripts.agent import build_trainer_from_config_path as build

In [ ]:
trainer = build("E:/Documents/Programming/Projects/Reinforcement/PhoenX_RL/src/Configs/actor_critic.yml")

In [ ]:
trainer.renderer

# Smooth Surrogate Testing

In [ ]:
import torch as T
import torch.nn.functional as F

def ppo_surrogate(ratio, advantages, epsilon=0.2, smooth=True, k=15.0):
    """
    ratio: tensor of π_new / π_old, shape advantages: tensor of A_t, shape epsilon: clip param, like 0.2
    smooth: bool, if True use soft clip instead of hard min/max
    k: sharpness of the soft curve (higher = sharper, like hard clip)
    
    Returns: clipped surrogate objective (scalar or per-sample)
    """
    if not smooth:
        # Vanilla PPO: hard clip
        clipped_ratio = T.clamp(ratio, 1 - epsilon, 1 + epsilon)
        surrogate = clipped_ratio * advantages
        return surrogate.mean()  # or .sum(), whatever your loss needs
    
    else:
        # Soft clip: gentle curve past bounds
        # We focus on upper bound here (r > 1+ε); lower is symmetric
        upper = 1 + epsilon
        print(f'upper:{upper}')
        # Softplus version: smooth ramp-down after upper
        # sigmoid(k*(r - upper)) starts at 1 when r=upper, drops to 0 as r grows
        decay = T.sigmoid(k * (ratio - upper))  # 0 to 1, smooth
        print(f'decay:{decay}')
        
        # Effective ratio: caps at upper, but adds a decaying tail
        effective_ratio = upper + (ratio - upper) * decay
        print(f'ratio-upper:{ratio-upper}')
        print(f'added ratio:{(ratio-upper)*decay}')
        print(f'effective ratio:{effective_ratio}')
        # Clamp lower too, for symmetry (optional but nice)
        effective_ratio = T.clamp(effective_ratio, 1 - epsilon, upper + 0.3)  # small overshoot
        print(f'effective lower clamp ratio:{effective_ratio}')
        surrogate = effective_ratio * advantages
        return surrogate.mean()

In [ ]:
advantages = T.tensor([1, 10, 20])
ratio = T.tensor([1.8])
epsilon = T.tensor([0.2])
k = 10

surrogates = ppo_surrogate(ratio, advantages, epsilon, k=k)
print(f'surrogates:{surrogates}')

In [ ]:
import torch

def ppo_surrogate(ratio, advantages, epsilon=0.2, smooth=False, k=12.0):
    """
    ratio: tensor of π_new / π_old
    advantages: tensor of A_t
    smooth=True → soft clipping (no zero gradients past bounds)
    k: higher = sharper decay (closer to hard clip)
    """
    if not smooth:
        # Original hard PPO clip
        clipped = T.clamp(ratio, 1 - epsilon, 1 + epsilon)
        return (clipped * advantages).mean()
    
    # === Soft version (only decays outside the clip window) ===
    lower = 1.0 - epsilon
    upper = 1.0 + epsilon
    effective = ratio.clone()
    
    # Upper side (r > 1+ε): overshoot a little, then decay back to 1+ε
    excess = T.clamp(ratio - upper, min=0.0)
    print(f'excess:{excess}')
    decay_upper = T.exp(-k * excess)
    print(f'decay_upper:{decay_upper}')
    effective = T.where(ratio > upper,
                            upper + excess * decay_upper,
                            effective)
    print(f'effective{effective}')
    
    # Lower side (r < 1-ε): undershoot a little, then decay back to 1-ε
    deficit = T.clamp(lower - ratio, min=0.0)
    print(f'deficit:{deficit}')
    decay_lower = T.exp(-k * deficit)
    print(f'decay_lower:{decay_lower}')
    effective = T.where(ratio < lower,
                            lower - deficit * decay_lower,
                            effective)
    print(f'effective{effective}')
    
    surrogate = effective * advantages
    return surrogate.mean()

In [ ]:
advantages = T.tensor([1])
ratio = T.tensor([0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6])
epsilon = T.tensor([0.2])
k = 15

surrogates = ppo_surrogate(ratio, advantages, epsilon, True, k=k)

In [ ]:
import torch as T

def ppo_surrogate(ratio, advantages, epsilon=0.2, smooth=False, k=10.0):
    """
    ratio: torch.Tensor of π_new / π_old
    advantages: torch.Tensor of A_t
    smooth=True → soft clipping (no zero gradients past bounds)
    k: controls how fast the overshoot tapers (higher k = closer to hard clip)
       Try 5–20. Default 10 is nice.
    """
    if not smooth:
        # Vanilla hard PPO clip
        clipped = T.clamp(ratio, 1 - epsilon, 1 + epsilon)
        return (clipped * advantages).mean()
    
    # === Monotonic soft clip (never decreases after crossing bound) ===
    lower = 1.0 - epsilon
    upper = 1.0 + epsilon
    effective = ratio.clone()
    
    # Upper side (r > 1+ε): gentle continued growth that slows down
    excess = T.clamp(ratio - upper, min=0.0)
    print(f'excess:{excess}')
    soft_excess = excess / (1.0 + k * excess)          # ← this is the key line
    print(f'soft_excess:{soft_excess}')
    effective = T.where(ratio > upper,
                            upper + soft_excess,
                            effective)
    print(f'effective:{effective}')
    
    # Lower side (r < 1-ε): symmetric
    deficit = T.clamp(lower - ratio, min=0.0)
    print(f'deficit:{deficit}')
    soft_deficit = deficit / (1.0 + k * deficit)
    print(f'soft_deficit:{soft_deficit}')
    effective = T.where(ratio < lower,
                            lower - soft_deficit,
                            effective)
    print(f'effective:{effective}')
    
    surrogate = effective * advantages
    return surrogate.mean()

In [ ]:
advantages = T.tensor([1])
ratio = T.tensor([0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8])
epsilon = T.tensor([0.2])
k = 15

surrogates = ppo_surrogate(ratio, advantages, epsilon, True, k=k)

In [7]:
import numpy as np
import torch as T
from app.env_wrapper import EnvWrapper
from dataclasses import dataclass
import gymnasium as gym
from gymnasium.vector import VectorEnv, SyncVectorEnv, VectorWrapper, utils
from typing import Optional, Dict, List
from collections import deque
from app.torch_utils import get_device
from app.utils import to_torch, to_numpy

In [8]:
@dataclass
class Observation:
    current_states: T.Tensor
    current_goals: T.Tensor | None = None
    current_ach_goals: T.Tensor | None = None
    transition_states: T.Tensor | None = None
    transition_goals: T.Tensor | None = None
    transition_ach_goals: T.Tensor | None = None
    rewards: T.Tensor | None = None
    terminations: T.Tensor | None = None
    truncations: T.Tensor | None = None
    infos: dict | None = None

In [19]:
class VectorNStepReward(VectorWrapper):
    def __init__(self, env, n: int, obs_key: str | None = None, goal_key: str | None = None, ach_goal_key: str | None = None):
        super().__init__(env)  # handles both gym.vector.VectorEnv and ManagerBasedRLEnv
        self.n = n
        self.obs_key = obs_key
        self.goal_key = goal_key
        self.ach_goal_key = ach_goal_key

        # Per-env deques (unchanged)
        self.n_states = [deque(maxlen=self.n) for _ in range(self.num_envs)]
        self.n_actions = [deque(maxlen=self.n) for _ in range(self.num_envs)]
        self.n_rewards = [deque(maxlen=self.n) for _ in range(self.num_envs)]
        self.n_next_states = [deque(maxlen=self.n) for _ in range(self.num_envs)]
        self.n_terminations = [deque(maxlen=self.n) for _ in range(self.num_envs)]
        self.n_truncations = [deque(maxlen=self.n) for _ in range(self.num_envs)]

        if self.goal_key:
            self.n_state_achieved_goals = [deque(maxlen=self.n) for _ in range(self.num_envs)]
            self.n_next_state_achieved_goals = [deque(maxlen=self.n) for _ in range(self.num_envs)]
            self.n_desired_goals = [deque(maxlen=self.n) for _ in range(self.num_envs)]

        self.current_states = None

        # Padding tensors (unchanged)
        self.device = get_device()
        state_shape = (self.single_observation_space[self.obs_key].shape
                       if self.obs_key is not None
                       else self.single_observation_space.shape)
        self._pad_state = T.zeros(state_shape, dtype=T.float32, device=self.device)
        self._pad_action = T.zeros(self.single_action_space.shape, dtype=T.float32, device=self.device)
        self._pad_reward = T.tensor(0.0, dtype=T.float32, device=self.device)
        self._pad_done = T.tensor(0.0, dtype=T.float32, device=self.device)
        if self.goal_key:
            self._pad_goal = T.zeros(self.single_observation_space[self.goal_key].shape, dtype=T.float32, device=self.device)

    def reset(self, **kwargs):
        states, infos = self.env.reset(**kwargs)
        # Clear env deques
        for i in range(self.num_envs):
            self.n_states[i].clear()
            self.n_actions[i].clear()
            self.n_rewards[i].clear()
            self.n_next_states[i].clear()
            self.n_terminations[i].clear()
            self.n_truncations[i].clear()
            if self.goal_key:
                self.n_state_achieved_goals[i].clear()
                self.n_next_state_achieved_goals[i].clear()
                self.n_desired_goals[i].clear()
        self.current_states = states
        infos.setdefault('n-step trajectory', {})
        return states, infos

    def step(self, actions: T.Tensor):
        next_states, rewards, terminations, truncations, infos = self.env.step(actions)
        # dones = terminations | truncations
        rewards = T.as_tensor(rewards, device=self.device)
        # dones = T.as_tensor(dones, device=device)
        actions = T.as_tensor(actions, device=self.device)
        terminations = T.as_tensor(terminations, device=self.device)
        truncations = T.as_tensor(truncations, device=self.device)

        dones = T.logical_or(terminations, truncations)

        for i in range(self.num_envs):
            # === IMPROVED FINAL-OBS HANDLING (robust for both old and new Gym/Isaac Lab keys) ===
            if dones[i].item():
                final_obs_key = None
                if "_final_obs" in infos and infos["_final_obs"][i]:
                    final_obs_key = "final_obs"
                elif "final_observation" in infos and infos.get("_final_observation", [False]*self.num_envs)[i]:
                    final_obs_key = "final_observation"
                elif "final_obs" in infos:
                    final_obs_key = "final_obs"

                if final_obs_key:
                    final = infos[final_obs_key][i]
                    next_state = final[self.obs_key] if self.obs_key is not None else final
                    if self.goal_key:
                        goal = self.current_states[self.goal_key][i]
                        ach_goal = self.current_states[self.ach_goal_key][i]
                        next_ach_goal = infos['final_obs'][i][self.ach_goal_key]
                else:
                    # fallback
                    next_state = next_states[self.obs_key][i] if self.obs_key is not None else next_states[i]
                    if self.goal_key:
                        goal = self.current_states[self.goal_key][i]
                        ach_goal = self.current_states[self.ach_goal_key][i]
                        next_ach_goal = next_states[self.ach_goal_key][i]
            else:
                next_state = next_states[self.obs_key][i] if self.obs_key is not None else next_states[i]
                if self.goal_key:
                    goal = self.current_states[self.goal_key][i]
                    ach_goal = self.current_states[self.ach_goal_key][i]
                    next_ach_goal = next_states[self.ach_goal_key][i]

            state = (self.current_states[self.obs_key][i]
                     if self.obs_key is not None
                     else self.current_states[i])

            # ensure tensors (unchanged)
            state = T.as_tensor(state, device=self.device)
            next_state = T.as_tensor(next_state, device=self.device)

            # Append current step
            self.n_states[i].append(state)
            self.n_actions[i].append(actions[i])
            self.n_rewards[i].append(rewards[i])
            self.n_next_states[i].append(next_state)
            self.n_terminations[i].append(terminations[i])
            self.n_truncations[i].append(truncations[i])
            if self.goal_key:
                self.n_state_achieved_goals[i].append(ach_goal if self.ach_goal_key is not None else None)
                self.n_next_state_achieved_goals[i].append(next_ach_goal if self.ach_goal_key is not None else None)
                self.n_desired_goals[i].append(goal if self.goal_key is not None else None)

        # Build batched trajectory (unchanged)
        trajectory = self.build_trajectories()
        infos['n-step trajectory'] = trajectory

        # Clear done trajectories
        for i in range(self.num_envs):
            if dones[i].item():
                self.n_states[i].clear()
                self.n_actions[i].clear()
                self.n_rewards[i].clear()
                self.n_next_states[i].clear()
                self.n_terminations[i].clear()
                self.n_truncations[i].clear()
                if self.goal_key:
                    self.n_state_achieved_goals[i].clear()
                    self.n_next_state_achieved_goals[i].clear()
                    self.n_desired_goals[i].clear()

        self.current_states = next_states
        return next_states, rewards, terminations, truncations, infos

    def build_trajectories(self):
        """Construct batched n-step trajectory dict from per-env deques."""
        states = self.format_trajectory(self.n_states, pad_mode="repeat")
        next_states = self.format_trajectory(self.n_next_states, pad_mode="repeat")
        actions = self.format_trajectory(self.n_actions, pad_mode="repeat")
        rewards = self.format_trajectory(self.n_rewards, pad_mode=T.tensor(0.0, dtype=T.float32, device=self.device))
        terminations = self.format_trajectory(self.n_terminations, pad_mode=T.tensor(0.0, dtype=T.float32, device=self.device))
        truncations = self.format_trajectory(self.n_truncations, pad_mode=T.tensor(0.0, dtype=T.float32, device=self.device))
        if self.goal_key:
            desired_goals = self.format_trajectory(self.n_desired_goals, pad_mode="repeat")
            state_achieved_goals = self.format_trajectory(self.n_state_achieved_goals, pad_mode="repeat")
            next_state_achieved_goals = self.format_trajectory(self.n_next_state_achieved_goals, pad_mode="repeat")

        # Determine actual trajectory lengths to compute n-step returns
        lengths = []
        for d in self.n_terminations:
            lengths.append(len(d))
        lengths = T.tensor(lengths, device=self.device)

        trajectory = {
            'states': states,
            'actions': actions,
            'rewards': rewards,
            'next_states': next_states,
            'terminations': terminations,
            'truncations': truncations,
            'trajectory_lengths': lengths,
        }
        if self.goal_key:
            trajectory['state_achieved_goals'] = state_achieved_goals
            trajectory['next_state_achieved_goals'] = next_state_achieved_goals
            trajectory['desired_goals'] = desired_goals
        return trajectory

    def format_trajectory(self, trajectory: List[deque[T.Tensor]], pad_mode:str|T.Tensor="repeat"):
        """Format trajectory from per-env deques to batched tensor.

        Args:
            trajectory: List of deques containing tensors.
            pad_mode: Mode to pad the trajectory. "repeat" to repeat the last value or tensor to pad with passed value.

        Returns:
            Tensor: Batched trajectory.
        """
        trajs = []
        for d in trajectory:
            seq = list(d)
            if pad_mode == "repeat":
                padding = seq[-1]
            else:
                padding = pad_mode
            while len(seq) < self.n:
                seq.append(padding)
            trajs.append(T.stack(to_torch(seq, device=self.device), dim=0))
        return T.stack(trajs, dim=0)

    @property
    def single_action_space(self):
        return self.env.single_action_space

    @property
    def single_observation_space(self):
        return self.env.single_observation_space

class OneHotObservationWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        assert isinstance(self.observation_space, gym.spaces.Discrete), "Observation space must be Discrete."
        self.observation_space = gym.spaces.Box(low=0.0, high=1.0, shape=(self.observation_space.n,), dtype=np.float32)
    
    def observation(self, obs):
        one_hot = np.zeros(self.observation_space.shape[0], dtype=np.float32)
        one_hot[obs] = 1.0
        return one_hot

class NumpyToTorch(VectorWrapper):
    def __init__(self, env, device=None):
        super().__init__(env)
        self.device = device
    def reset(self, *, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=to_numpy(options))
        return to_torch(obs, self.device), to_torch(info, self.device)
    def step(self, actions):
        obs, reward, terminated, truncated, info = self.env.step(to_numpy(actions))
        return (
            to_torch(obs, self.device),
            to_torch(reward, self.device),
            to_torch(terminated, self.device),
            to_torch(truncated, self.device),
            to_torch(info, self.device),
        )
    def render(self):
        return self.env.render()


WRAPPER_REGISTRY = {
    # "AtariPreprocessing": {
    #     "cls": gym_wrappers.AtariPreprocessing,
    #     "default_params": {
    #         "frame_skip": 1,
    #         "grayscale_obs": True,
    #         "scale_obs": True
    #     }
    # },
    # "TimeLimit": {
    #     "cls": gym_wrappers.TimeLimit,
    #     "default_params": {
    #         "max_episode_steps": 1000
    #     }
    # },
    # "TimeAwareObservation": {
    #     "cls": gym_wrappers.TimeAwareObservation,
    #     "default_params": {
    #         "flatten": False,
    #         "normalize_time": False
    #     }
    # },
    # "FrameStackObservation": {
    #     "cls": gym_wrappers.FrameStackObservation,
    #     "default_params": {
    #         "stack_size": 4
    #     }
    # },
    # "ResizeObservation": {
    #     "cls": gym_wrappers.ResizeObservation,
    #     "default_params": {
    #         "shape": 84
    #     }
    # },
    # "NStepReward": {
    #     "cls": NStepReward,
    #     "default_params": {"n": 1}
    # },
    "VectorNStepReward": {
        "cls": VectorNStepReward,
        "vector_aware": True,
        "default_params": {"n": 1, "obs_key": None, "goal_key": None, "ach_goal_key": None}
    },
    "OneHotObservationWrapper": {
        "cls": OneHotObservationWrapper,
        "default_params": {}
    }
}

In [ ]:
class GymnasiumWrapper(EnvWrapper):
    """
    Wrapper for Gymnasium environments with additional utilities.

    This wrapper supports initialization, resetting, stepping, rendering,
    and JSON-based serialization of Gymnasium environments.
    """
    def __init__(self, cfg:str, num_envs:int=1, wrappers:list[dict]|None=None,
                 render_mode:str|None=None, seed:int|None=None, obs_key:str|None=None, goal_key:str|None=None, ach_goal_key:str|None=None):
        self.env_id = cfg
        self.wrappers = wrappers
        self.num_envs = num_envs
        if seed is None:
            seed = T.randint(2**31-1, (1,)).item()
        self.seed = seed
        self.render_mode = render_mode
        self.obs_key = obs_key
        self.goal_key = goal_key
        self.ach_goal_key = ach_goal_key
        self.env = self._initialize_env()
        

    def _initialize_env(self):
        """
        Initialize the Gymnasium environments.

        
        Returns:
            gym.VectorEnv: The initialized Gymnasium vectorized environment.
        """
        single_wrappers = []
        vector_wrappers = []
        if self.wrappers:
            for wrapper in self.wrappers:
                wrapper_type = wrapper.get('type')
                if not wrapper_type:
                    raise ValueError("Each wrapper dict must have a 'type' key.")
                
                if wrapper_type in WRAPPER_REGISTRY:
                    entry = WRAPPER_REGISTRY[wrapper_type]
                    cls = entry["cls"]
                    default_params = entry["default_params"].copy()
                    vector_aware = entry.get("vector_aware", False)
                else:
                    # Dynamic resolution for built-in Gymnasium wrappers
                    if hasattr(gym_vector_wrappers, wrapper_type):
                        cls = getattr(gym_vector_wrappers, wrapper_type)
                        vector_aware = True
                    elif hasattr(gym_wrappers, wrapper_type):
                        cls = getattr(gym_wrappers, wrapper_type)
                        vector_aware = False
                    else:
                        raise ValueError(f"Unknown wrapper type '{wrapper_type}'. Add to WRAPPER_REGISTRY or ensure it's a valid Gymnasium wrapper class name.")
                    
                    default_params = {}  # No defaults for unresolved; rely on user params
                
                override_params = wrapper.get("params", {})
                final_params = {**default_params, **override_params}
                # final_params.update({"obs_key": self.obs_key, "goal_key": self.goal_key})
                
                if vector_aware:
                    vector_wrappers.append((cls, final_params))
                else:
                    def wrapper_fn(env, cls=cls, params=final_params):
                        return cls(env, **params)
                    single_wrappers.append(wrapper_fn)

        # Create vector env with single-env wrappers applied per sub-env
        vec_env = gym.make_vec(
            id=self.env_id,
            num_envs=self.num_envs,
            vectorization_mode="sync",
            vector_kwargs={"autoreset_mode": "SameStep"},
            wrappers=single_wrappers,
            render_mode=self.render_mode
        )

        # Apply vector-aware wrappers to the entire vec_env
        for cls, params in vector_wrappers:
            vec_env = cls(vec_env, **params)

        # Wrap vectorized environment to return tensors
        vec_env = NumpyToTorch(vec_env, device=get_device())

        return vec_env

    def extract_states_goals(
        self,
        states: np.ndarray | T.Tensor | dict | list[dict]
    )->tuple[T.Tensor, T.Tensor | None, T.Tensor | None]:
        """Extract the states and goals from the passed states argument and returns them as Tensors.
        
        Args:
            states (np.ndarray | T.Tensor | dict | list[dict]): States to extract from.
        
        Returns:
            tuple: Tuple of states, goals, and achieved goals as Tensors.
        """
        device = get_device()
        if isinstance(states, list):
            obs_list = []
            goals_list = []
            ach_goals_list = []
            for step_data in states:
                if isinstance(step_data, dict):
                    if not self.obs_key:
                        raise ValueError("Goal-aware observation spaces require obs_key to be set")
                    step_obs = step_data.get(self.obs_key)
                    if self.goal_key:
                        step_goal = step_data.get(self.goal_key)
                    else:
                        step_goal = None
                    if self.ach_goal_key:
                        step_ach_goal = step_data.get(self.ach_goal_key)
                    else:
                        step_ach_goal = None
                else:
                    break

                # Convert to tensor if needed
                if not isinstance(step_obs, T.Tensor):
                    step_obs = T.tensor(step_obs, dtype=T.float32, device=device)
                if self.goal_key and step_goal is not None:
                    if not isinstance(step_goal, T.Tensor):
                        step_goal = T.tensor(step_goal, dtype=T.float32, device=device)
                    goals_list.append(step_goal)
                if self.ach_goal_key and step_ach_goal is not None:
                    if not isinstance(step_ach_goal, T.Tensor):
                        step_ach_goal = T.tensor(step_ach_goal, dtype=T.float32, device=device)
                    ach_goals_list.append(step_ach_goal)
                obs_list.append(step_obs)
            obs = T.stack(obs_list, dim=0)
            
            if self.goal_key:
                goals = T.stack(goals_list, dim=0)
            else:
                goals = None
            if self.ach_goal_key:
                ach_goals = T.stack(ach_goals_list, dim=0)
            else:
                ach_goals = None

        elif isinstance(states, dict):
            if not self.obs_key:
                raise ValueError("Goal-aware observation spaces require obs_key to be set")
            obs = states.get(self.obs_key)
            if self.goal_key:
                goals = states.get(self.goal_key)
            else:
                goals = None
            if self.ach_goal_key:
                ach_goals = states.get(self.ach_goal_key)
            else:
                ach_goals = None
        else:
            obs = states
            goals = None
            ach_goals = None

        if not isinstance(obs, T.Tensor):
            obs = T.tensor(obs, dtype=T.float32, device=device)
        if goals is not None and not isinstance(goals, T.Tensor):
            goals = T.tensor(goals, dtype=T.float32, device=device)
        if ach_goals is not None and not isinstance(ach_goals, T.Tensor):
            ach_goals = T.tensor(ach_goals, dtype=T.float32, device=device)
        
        return obs, goals, ach_goals

    def render_frame(self)->np.ndarray:
        """Renders a frame from the environment.
        
        Returns:
            np.ndarray: The rendered frame.
        """
        frame = self.env.render()        
        return frame[0]
        

    def reset(self, seed:int|None=None):
        if seed is not None:
            effective_seed = seed
        else:
            effective_seed = self.seed

        states, infos = self.env.reset(seed=effective_seed)
        self.env.action_space.seed(seed=effective_seed)
        
        obs, goals, ach_goals = self.extract_states_goals(states)
        
        return Observation(
            current_states=obs,
            current_goals=goals,
            current_ach_goals=ach_goals,
            transition_states=obs,
            transition_goals=goals,
            transition_ach_goals=ach_goals,
            infos=infos
        )

    def step(self, action)->Observation:
        states, rewards, terminations, truncations, infos = self.env.step(action)

        # Set initial value of transition states to states
        transition_states = states.clone() if isinstance(states, T.Tensor) else states.copy()
        # If env terminated/truncated, get terminal state and set as transition
        if "final_obs" in infos:
            if self.num_envs > 1:
                mask = infos.get("_final_obs").cpu()
                term_states = np.stack(infos["final_obs"][mask])
                transition_states[mask] = T.as_tensor(term_states, dtype=states.dtype, device=states.device)
            else:
                term_states = infos["final_obs"][0]
                transition_states = T.as_tensor(term_states, dtype=states.dtype, device=states.device)

        # Separate observations, goals, and achieved goals 
        obs, goals, ach_goals = self.extract_states_goals(states)
        transition_obs, transition_goals, transition_ach_goals = self.extract_states_goals(transition_states)

        return Observation(
            current_states=obs,
            current_goals=goals,
            current_ach_goals=ach_goals,
            transition_states=transition_obs,
            transition_goals=transition_goals,
            transition_ach_goals=transition_ach_goals,
            rewards=rewards,
            terminations=terminations,
            truncations=truncations,
            infos=infos
        )

    def sample_observation(self):
        actions = self.action_space.sample()
        observation = self.step(actions)
        obs, goals, ach_goals = self.extract_states_goals(observation.current_states)
        return Observation(
            current_states=obs,
            current_goals=goals,
            current_ach_goals=ach_goals
        )
    
    def format_actions(self, actions: np.ndarray | T.Tensor):
        if isinstance(actions, T.Tensor):
            actions = actions.cpu().numpy()
        if isinstance(self.action_space, gym.spaces.Box):
            # if testing:
            #     num_envs = 1
            # else:
            num_envs = self.env.num_envs
            num_actions = self.action_space.shape[-1]
            return actions.reshape(num_envs, num_actions)
        if isinstance(self.action_space, gym.spaces.Discrete) or isinstance(self.action_space, gym.spaces.MultiDiscrete):
            return actions.ravel()
        
    def get_base_env(self):
        """Recursively unwrap an environment to get the base environment."""
        env = self.env.env
        while hasattr(env, 'env'):
            env = env.env
        return env
    
    def close(self):
        """
        Close the environment.
        """
        self.env.close()
    
    @property
    def observation_space(self):
        """
        Get the observation space of the environment.

        Returns:
            gym.Space: The observation space.
        """
        return self.env.observation_space
    
    @property
    def action_space(self):
        """
        Get the action space of the environment.

        Returns:
            gym.Space: The action space.
        """
        return self.env.action_space
    
    @property
    def single_action_space(self):
        """
        Get the single action space for vectorized environments.

        Returns:
            gym.Space: The single action space.
        """
        return self.env.single_action_space

    @property
    def single_observation_space(self):
        """
        Get the single observation space for vectorized environments.

        Returns:
            gym.Space: The single observation space.
        """
        return self.env.single_observation_space

    @property
    def finite_horizon(self)->bool:
        """
        Returns True if the environment has a finite horizon.
        Finite horizon is determined by checking if the base environment spec contains has
        a max_episode_steps attribute that is not None, or if the environment is wrapped in a 
        TimeLimit wrapper.
        """
        base_env = self.get_base_env()
        if hasattr(base_env, 'spec') and base_env.spec is not None:
            return base_env.spec.max_episode_steps is not None

        env = self.env
        while hasattr(env, 'env'):
            if isinstance(env, gym.wrappers.TimeLimit):
                return True
            env = env.env
        
        return False
    
    @property
    def config(self):
        """
        Get the configuration of the wrapper.

        Returns:
            dict: Configuration dictionary.
        """
        return {
            "type": "gymnasium",
            "config":{
                "cfg": self.env_id,
                "num_envs": self.num_envs,
                "wrappers": self.wrappers,
                "render_mode": self.render_mode,
                "seed": self.seed,
                "obs_key": self.obs_key,
                "goal_key": self.goal_key,
                "ach_goal_key": self.ach_goal_key,
            }
        }
    
    def to_json(self):
        """
        Serialize the wrapper configuration to JSON.

        Returns:
            str: JSON string representing the configuration.
        """
        return json.dumps(self.config)

    @classmethod
    def from_json(cls, json_env_spec):
        """
        Create a Gymnasium wrapper instance from a JSON string.

        Args:
            json_env_spec (str): JSON string representing the configuration.

        Returns:
            GymnasiumWrapper: A new Gymnasium wrapper instance.
        """
        config = json.loads(json_env_spec)
        config = config['config']
        try:
            return cls(**config)
        except Exception as e:
            raise ValueError(f"Environment wrapper error: {config}, {e}")

In [27]:
# Create Env
env = GymnasiumWrapper(
    cfg='FetchPickAndPlaceDense-v4',
    num_envs=2,
    wrappers=[{'type':'VectorNStepReward', 'params':{'n':3, 'obs_key':'observation', 'goal_key':'desired_goal', 'ach_goal_key':'achieved_goal'}}],
    render_mode=None,
    seed=42,
    obs_key='observation',
    goal_key='desired_goal',
    ach_goal_key='achieved_goal'
    )

In [28]:
obs = env.reset()
print(obs)

Observation(current_states=tensor([[ 1.3419e+00,  7.4910e-01,  5.3473e-01,  1.4495e+00,  8.0831e-01,
          4.2470e-01,  1.0758e-01,  5.9210e-02, -1.1002e-01,  3.8359e-06,
          6.4454e-08, -2.3028e-18, -1.1741e-12,  5.9237e-17,  5.2614e-06,
          7.5003e-08,  2.2207e-05,  2.6948e-18,  6.7047e-13, -1.2291e-19,
         -5.2614e-06, -7.5003e-08,  2.4621e-05, -9.8816e-07,  7.6532e-08],
        [ 1.3419e+00,  7.4910e-01,  5.3473e-01,  1.3876e+00,  6.1223e-01,
          4.2470e-01,  4.5690e-02, -1.3687e-01, -1.1002e-01,  3.8359e-06,
          6.4454e-08, -2.3028e-18, -1.1741e-12,  5.9237e-17,  5.2614e-06,
          7.5003e-08,  2.2207e-05,  2.6948e-18,  6.7047e-13, -1.2291e-19,
         -5.2614e-06, -7.5003e-08,  2.4621e-05, -9.8816e-07,  7.6532e-08]],
       device='cuda:0'), current_goals=tensor([[1.2202, 0.8918, 0.4247],
        [1.1979, 0.8509, 0.7630]], device='cuda:0'), current_ach_goals=tensor([[1.4495, 0.8083, 0.4247],
        [1.3876, 0.6122, 0.4247]], device='cuda:0'),

In [29]:
while True:
    obs = env.step(env.action_space.sample())
    if 'final_obs' in obs.infos:
        break

In [ ]:
# If env terminated/truncated, get terminal state and set as transition
        # if "final_obs" in infos:
        #     if self.num_envs > 1:
        #         mask = infos.get("_final_obs").cpu()
        #         term_states = np.stack(infos["final_obs"][mask])
        #         # if isinstance(transition_states, T.Tensor):
        #         transition_states[mask] = T.as_tensor(term_states, dtype=states.dtype, device=states.device)
        #         # else:
        #         #     for key in transition_states.keys():
        #         #         transition_states[key][mask] = T.as_tensor(term_states[key], dtype=states[key].dtype, device=states[key].device)
        #     else:
        #         term_states = infos["final_obs"][0]
        #         transition_states = T.as_tensor(term_states, dtype=states.dtype, device=states.device)

In [30]:
obs.infos

{'final_obs': array([{'observation': array([ 1.41339435e+00,  8.12074210e-01,  4.16598486e-01,  1.49168101e+00,
                8.41416961e-01,  4.24851554e-01,  7.82866547e-02,  2.93427506e-02,
                8.25306806e-03,  3.94620971e-02,  3.90180400e-02,  1.51370005e-03,
                7.47621278e-05, -7.34610858e-01,  6.16786353e-03,  5.23568024e-03,
                1.20249863e-02, -4.93252408e-02, -2.75477681e-02, -7.49423150e-02,
                3.71294817e-03,  5.20054512e-03, -1.27292032e-02, -1.02272907e-02,
               -6.99809674e-03]), 'achieved_goal': array([1.49168101, 0.84141696, 0.42485155]), 'desired_goal': array([1.22018822, 0.89178776, 0.42469975])},
        {'observation': array([ 1.21391547e+00,  8.79570578e-01,  5.59681075e-01,  1.38762480e+00,
                6.12233649e-01,  4.24784489e-01,  1.73709325e-01, -2.67336929e-01,
               -1.34896586e-01,  7.80481489e-04,  2.30306503e-04,  9.36725283e-17,
               -3.12668709e-19,  5.83608118e-17, -

In [32]:
mask = obs.infos.get("_final_obs").cpu()
print(mask)

tensor([True, True])


In [33]:
term_states = np.stack(obs.infos["final_obs"][mask])
print(term_states)

[{'observation': array([ 1.41339435e+00,  8.12074210e-01,  4.16598486e-01,  1.49168101e+00,
         8.41416961e-01,  4.24851554e-01,  7.82866547e-02,  2.93427506e-02,
         8.25306806e-03,  3.94620971e-02,  3.90180400e-02,  1.51370005e-03,
         7.47621278e-05, -7.34610858e-01,  6.16786353e-03,  5.23568024e-03,
         1.20249863e-02, -4.93252408e-02, -2.75477681e-02, -7.49423150e-02,
         3.71294817e-03,  5.20054512e-03, -1.27292032e-02, -1.02272907e-02,
        -6.99809674e-03]), 'achieved_goal': array([1.49168101, 0.84141696, 0.42485155]), 'desired_goal': array([1.22018822, 0.89178776, 0.42469975])}
 {'observation': array([ 1.21391547e+00,  8.79570578e-01,  5.59681075e-01,  1.38762480e+00,
         6.12233649e-01,  4.24784489e-01,  1.73709325e-01, -2.67336929e-01,
        -1.34896586e-01,  7.80481489e-04,  2.30306503e-04,  9.36725283e-17,
        -3.12668709e-19,  5.83608118e-17, -1.98000013e-02,  1.28908971e-02,
         3.13512147e-03, -4.61792723e-17, -1.10298046e-17,

In [ ]:
transition_states = states.clone() if isinstance(states, T.Tensor) else states.copy()